# PA1 — Task 3: Domain Generalization on PACS

**The experiment in one line.** Same sources (Photo, Art Painting, Cartoon), same target (Sketch),
but now **nothing** may look at Sketch until the very last script. The question is whether anything
beats plain supervised training when the target domain is genuinely unavailable.

| Run | Idea |
|---|---|
| `erm` | plain supervised training — **the Task 2 Source-only checkpoint, reused, not retrained** |
| `dan_dg` | make the three *source* domains indistinguishable from each other (pairwise MMD), hoping that also covers Sketch |
| `sam` | find parameters whose source loss is locally stable (flat), via two forward/backward passes per update |

Plus two runs for the controlled study (λ_DG = 0.1 and 10), which mirrors Task 2's DAN λ study and
lets you compare *the same penalty with and without target access*.

**Two hard rules this notebook enforces**
1. **No Sketch anywhere except Part D.** `task3/train.py` never even imports the Sketch loader.
   Part C evaluates the source side only, and you look at it before Part D exists.
2. **Same protocol as Task 2.** `task3/train.py` refuses to start if the settings differ from those
   stored with the ERM checkpoint, because ERM *is* that checkpoint.

**Before you start**
1. Task 2 must be finished, with `task2/results/runs/source_only/best.pt` in place.
2. Upload `pa1-task3.zip` to **My Drive**.
3. **Runtime → Change runtime type → T4 GPU → Save**.

## Part A — setup (run every session)

In [ ]:
# A1. Check the GPU.
import subprocess
out = subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader',
                     shell=True, capture_output=True, text=True).stdout.strip()
print(f'GPU: {out}' if out else 'NO GPU -- Runtime > Change runtime type > T4 GPU, then re-run.')

In [ ]:
# A2. Mount Drive.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# A3. Install the Task 3 files into your repository (new files only; nothing existing is overwritten
# except the two READMEs, which gain the Task 3 entries).
import os, shutil
DRIVE = '/content/drive/MyDrive'
REPO = f'{DRIVE}/PA1'                        # <- change if your repo folder has another name
ZIP = f'{DRIVE}/pa1-task3.zip'
assert os.path.isdir(REPO), f'Repository not found at {REPO}'
assert os.path.exists(ZIP), f'Upload pa1-task3.zip to My Drive first (expected {ZIP})'
!unzip -q -n "{ZIP}" -x README.md -d "{REPO}"
!unzip -q -o "{ZIP}" README.md -d "{REPO}"
%cd {REPO}
!ls task3

In [ ]:
# A4. Helpers. `report()` here reads Task 3 runs and knows their extra keys
# (loss_mmd for DAN-DG, loss_sam for SAM).
import subprocess, json, math, glob
import pandas as pd
import matplotlib.pyplot as plt

CHANCE_LOSS = math.log(7)     # 1.946 = cross-entropy of uniform guessing over 7 classes

def run(cmd):
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    if p.wait() != 0:
        raise RuntimeError(f'Command failed (exit {p.returncode}): {cmd}')

def report(run_name):
    """Health summary + curves for one Task 3 run. What to check:
         loss_cls  -- must end far below 1.95; stuck near it means the classifier collapsed
         loss_mmd  -- DAN-DG's source-alignment penalty. The biased 8-vs-8 estimate cannot go
                      below ~0.47 even for identical domains, and that offset is not harmless:
                      its gradient pulls features smaller (see shared/mmd.py)
         loss_sam  -- SAM's loss at the perturbed point; the gap to loss_cls is the local
                      sharpness the optimiser is actually seeing during training
    """
    d = f'task3/results/runs/{run_name}'
    s, hist = json.load(open(f'{d}/summary.json')), json.load(open(f'{d}/history.json'))
    ep = [h['epoch'] for h in hist]
    f1 = [h['val_mean_macro_f1'] for h in hist]
    cls = [h['loss_cls'] for h in hist]
    best_ep = ep[int(pd.Series(f1).idxmax())]
    print(f'\n=== {run_name} ===')
    print(f'  epochs run              : {s["epochs_run"]} of {s["config"]["max_epochs"]}'
          f' ({"early stop" if s["epochs_run"] < s["config"]["max_epochs"] else "full budget"})')
    print(f'  best source-val macro-F1: {s["best_mean_macro_f1"]:.2f}  (epoch {best_ep} -- checkpoint saved there)')
    print(f'  classification loss     : {cls[0]:.3f} -> {cls[-1]:.3f}   (1.95 = chance)')
    if 'loss_mmd' in hist[0]:
        print(f'  source-pair MMD penalty : {hist[0]["loss_mmd"]:.3f} -> {hist[-1]["loss_mmd"]:.3f}   (trend only)')
    if 'loss_sam' in hist[0]:
        gap0, gap1 = hist[0]['loss_sam'] - hist[0]['loss_cls'], hist[-1]['loss_sam'] - hist[-1]['loss_cls']
        print(f'  SAM perturbed loss      : {hist[0]["loss_sam"]:.3f} -> {hist[-1]["loss_sam"]:.3f}'
              f'   (gap above loss_cls: {gap0:.3f} -> {gap1:.3f})')
    gn = [h.get('grad_norm', float('nan')) for h in hist]
    clip = s['config'].get('grad_clip', 0.0)
    if clip:
        print(f'  gradient norm           : max {max(gn):.2f}, clipping (at {clip}) active on '
              f'{sum(g > clip for g in gn if g == g)}/{len(gn)} epochs')
    print('  VERDICT: ' + ('COLLAPSED -- never learned the classes; send it to Claude.'
                           if min(cls) > 1.6 or s['best_mean_macro_f1'] < 50 else
                           'unstable -- learned then degraded; look before continuing.'
                           if cls[-1] > 1.6 or s['best_mean_macro_f1'] < 80 else 'looks healthy.'))
    keys = [('loss_cls', 'classification loss', CHANCE_LOSS)]
    if 'loss_mmd' in hist[0]: keys.append(('loss_mmd', 'source-pair MMD', None))
    if 'loss_sam' in hist[0]: keys.append(('loss_sam', 'loss at theta+eps (SAM)', None))
    keys.append(('val_mean_macro_f1', 'mean source-val macro-F1', None))
    fig, axes = plt.subplots(1, len(keys), figsize=(3.2 * len(keys), 2.6))
    for ax, (k, title, line) in zip(axes, keys):
        ax.plot(ep, [h.get(k, float('nan')) for h in hist], marker='.')
        if line: ax.axhline(line, color='r', ls=':', lw=1)
        ax.set_title(f'{run_name}: {title}', fontsize=8); ax.set_xlabel('epoch'); ax.grid(alpha=.3)
    fig.tight_layout(); plt.show()

print('Helpers ready: run(cmd), report(run_name)')

In [ ]:
# A5. PACS on local disk (restored from the Drive backup you made in Task 2).
PACS = '/content/data/PACS'
DRIVE_ZIP = f'{DRIVE}/datasets/PACS.zip'
EXPECTED = {'photo': 1670, 'art_painting': 2048, 'cartoon': 2344, 'sketch': 3929}
if not os.path.isdir(f'{PACS}/sketch'):
    assert os.path.exists(DRIVE_ZIP), 'No PACS backup found -- re-run the Task 2 notebook cell A5 once.'
    print('Restoring PACS from Drive ...')
    run(f'unzip -q "{DRIVE_ZIP}" -d /content/data')
for d, n in EXPECTED.items():
    got = len([p for p in glob.glob(f'{PACS}/{d}/*/*') if p.lower().endswith(('.jpg', '.jpeg', '.png'))])
    print(f'  {d:13s} {got:5d}  (expected {n})' + ('' if got == n else '   <-- MISMATCH'))
print('\nThe sketch folder is present on disk, but no Task 3 training or diagnostic script opens it.')

In [ ]:
# A6. Check the ERM baseline: it must be the Task 2 Source-only checkpoint, and Task 3 must use the
# same protocol. Whatever Task 2 finally ran with (clipping on or off) is what Task 3 inherits.
from common.io import load_config, load_json
ck = 'task2/results/runs/source_only/best.pt'
assert os.path.exists(ck), 'Task 2 Source-only checkpoint missing -- finish Task 2 first.'
erm_cfg = load_json('task2/results/runs/source_only/config_resolved.json')
t3 = load_config('task3/configs/base.yaml')
KEYS = ['seed', 'num_workers', 'per_domain_batch', 'lr', 'weight_decay', 'grad_clip', 'max_epochs', 'patience']
rows = [{'setting': k, 'ERM checkpoint': erm_cfg.get(k), 'Task 3': t3.get(k),
         'match': 'yes' if erm_cfg.get(k) == t3.get(k) else 'NO'} for k in KEYS]
display(pd.DataFrame(rows))
src_f1 = json.load(open('task2/results/runs/source_only/summary.json'))['best_mean_macro_f1']
print(f'ERM (reused, not retrained): best mean source-val macro-F1 = {src_f1:.2f}')
print('Any row marked NO must be fixed before training, or ERM and the Task 3 methods are not comparable.')

## Part B — training (ERM is not trained; it is loaded)

**What one update looks like.** 8 photo + 8 art + 8 cartoon images, cross-entropy over all 24, then:
- **DAN-DG** adds (λ_DG/3) × the MMD between each of the three source pairs, computed from 8 + 8
  features with a per-pair median bandwidth.
- **SAM** instead does two passes: compute the gradient, step *uphill* by ρ = 0.05 in its direction,
  recompute the gradient there, step back, then let AdamW use that second gradient.

Everything else is Task 2's protocol, unchanged.

**Before running:** write down what you expect increasing λ_DG to do to source performance, to
source-domain separability, and to Sketch. The handout requires the expectation to be stated first,
and you may not swap the main setting (λ_DG = 1, ρ = 0.05) for a post-hoc winner.

In [ ]:
# B0. Run list + trainer. Finished runs are skipped, so this can be spread over sessions.
STUDY = 'dan_dg'      # 'dan_dg' -> lambda_DG in {0.1, 1, 10}   (recommended: directly comparable
                      #             with Task 2's DAN lambda study, i.e. same penalty, no target)
                      # 'sam'    -> rho in {0.01, 0.05, 0.1}
COMMON = f'--set data_root={PACS}'
RUNS = [('dan_dg', 'dan_dg'), ('sam', 'sam')]
RUNS += ([('study_dan_dg_lambda0.1', 'dan_dg_lambda0.1'), ('study_dan_dg_lambda10', 'dan_dg_lambda10')]
         if STUDY == 'dan_dg' else
         [('study_sam_rho0.01', 'sam_rho0.01'), ('study_sam_rho0.1', 'sam_rho0.1')])

def train(config, run_name):
    if os.path.exists(f'task3/results/runs/{run_name}/summary.json'):
        print(f'{run_name}: already finished -- skipping')
    else:
        run(f'python -m task3.train --config task3/configs/{config}.yaml {COMMON}')
    report(run_name)

print(f'Controlled study: {STUDY}. Runs to train (ERM is reused, not trained):\n')
for c, r in RUNS:
    print(f'  {"done   " if os.path.exists(f"task3/results/runs/{r}/summary.json") else "pending"}  {r}')

In [ ]:
# B1. DAN-DG -- pairwise MMD between photo, art and cartoon features. No Sketch involved.
# Expect: loss_cls similar to ERM, loss_mmd drifting down. The open question is whether invariance
# across these three domains transfers to a fourth one it never saw.
train('dan_dg', 'dan_dg')

In [ ]:
# B2. SAM -- two passes per update, rho = 0.05. Roughly twice the compute per step.
# Expect: loss_sam slightly above loss_cls, with the gap shrinking as the solution flattens.
train('sam', 'sam')

In [ ]:
# B3 + B4. The controlled study: the two extra settings (the middle one is the main run above).
for c, r in RUNS[2:]:
    train(c, r)

In [ ]:
# B5. All Task 3 runs side by side (source information only).
rows = []
for c, r in RUNS:
    p = f'task3/results/runs/{r}/summary.json'
    if not os.path.exists(p):
        rows.append({'run': r, 'status': 'NOT RUN'}); continue
    s, hist = json.load(open(p)), json.load(open(f'task3/results/runs/{r}/history.json'))
    cls = [e['loss_cls'] for e in hist]
    rows.append({'run': r, 'status': 'COLLAPSED' if min(cls) > 1.6 or s['best_mean_macro_f1'] < 50
                 else 'unstable' if cls[-1] > 1.6 or s['best_mean_macro_f1'] < 80 else 'ok',
                 'epochs': s['epochs_run'], 'best_src_val_f1': round(s['best_mean_macro_f1'], 2),
                 'final_loss_cls': round(cls[-1], 3)})
display(pd.DataFrame(rows))
print(f'ERM reference (Task 2 Source-only): best mean source-val macro-F1 = {src_f1:.2f}')

## Part C — source-side evaluation (still no Sketch)

This is the part that makes the protocol checkable. It reports, for ERM, DAN-DG and SAM:

- accuracy and macro-F1 per source domain, plus **mean** and **worst** domain;
- **source-domain separability**: freeze the backbone, take balanced features from the three source
  *validation* sets, 70/30 split at seed 6304, multinomial logistic regression at C = 1 predicting
  photo / art / cartoon. Chance is 33.3%. Lower means the sources are harder to tell apart;
- the **shared sharpness proxy**: on one fixed batch of 32 images per source (seed 6304), in eval
  mode, take one ascent step of radius 0.05 and measure how much the loss rises. Lower means locally
  more stable *under this specific perturbation*, not globally flatter.

Look at all of it now. Once Part D runs, nothing here may change.

In [ ]:
# C1. Source-side results for the three main models, then for the study.
run(f'python -m task3.evaluate_sources {COMMON}')
run(f'python -m task3.evaluate_sources --study {STUDY} {COMMON}')

F3 = 'task3/results/final'
ss = pd.read_csv(f'{F3}/source_side.csv')
display(ss.round(3))
base = ss[ss.method == 'erm'].iloc[0]
for _, r in ss.iterrows():
    print(f'  {r.method:8s} mean {r.mean_acc:5.2f}% | worst {r.worst_acc:5.2f}% ({["photo","art_painting","cartoon"][int(pd.Series([r.photo_acc, r.art_painting_acc, r.cartoon_acc]).idxmin())]})'
          f' | source separability {r.src_domain_sep:5.2f}% (33.3 = chance) | sharpness {r.sharpness:+.4f}')
print('\nThree questions to answer here, all without Sketch:')
print('  - did DAN-DG actually reduce source separability relative to ERM?')
print('  - did SAM actually reduce the sharpness proxy relative to ERM and DAN-DG?')
print('  - does mean-source hide a weak domain (compare mean with worst)?')

## Part D — Sketch evaluation (the only Sketch access in Task 3)

Run this once, after Part C has been read and nothing else will change. It reports Sketch accuracy
and macro-F1, the change versus ERM, per-class changes, failure cases, and the Task 2 vs Task 3
comparison that research question 4 asks for.

In [ ]:
# D1. Sketch evaluation for the main comparison and the study.
run(f'python -m task3.evaluate_sketch {COMMON}')
run(f'python -m task3.evaluate_sketch --study {STUDY} {COMMON}')

main = pd.read_csv(f'{F3}/table_main.csv')
display(main.round(2))
erm = main[main.method == 'erm'].iloc[0]
print('\n================ TASK 3 MAIN COMPARISON ================')
for _, r in main.iterrows():
    tag = 'baseline' if r.method == 'erm' else f'{r.sketch_acc - erm.sketch_acc:+.2f} pts vs ERM'
    print(f'  {r.method:8s} source mean {r.mean_f1:5.2f} F1 / worst {r.worst_f1:5.2f} | Sketch {r.sketch_acc:5.2f}% '
          f'| src-sep {r.src_domain_sep:5.2f}% | sharpness {r.sharpness:+.4f}   ({tag})')
print('\nRQ1: how well did mean-source and worst-source predict this Sketch ranking?')
print('RQ2: did lower source separability come with better Sketch accuracy?')
print('RQ3: does the sharpness ranking agree with the Sketch ranking?')

In [ ]:
# D2. The controlled study.
st = pd.read_csv(f'{F3}/table_study_{STUDY}.csv')
display(st.round(3))
for _, r in st.iterrows():
    print(f'  {r.setting:12s} source mean {r.mean_f1:5.2f} F1 | src-sep {r.src_domain_sep:5.2f}% '
          f'| sharpness {r.sharpness:+.4f} | Sketch {r.sketch_acc:5.2f}%')
best_src = st.loc[st.mean_f1.idxmax()]
print(f'\nChosen WITHOUT Sketch (highest mean source-val F1): {best_src.setting}')
print(f'Best on Sketch, known only now                     : {st.loc[st.sketch_acc.idxmax()].setting}')
print('The main comparison keeps lambda_DG = 1 / rho = 0.05 regardless of what won here.')

In [ ]:
# D3. Per-class Sketch changes and failure analysis.
pc = pd.read_csv(f'{F3}/per_class_sketch_acc.csv', index_col=0)
print('Per-class Sketch accuracy (%):'); display(pc.round(2))
print('Change versus ERM (percentage points):')
display(pc.drop(columns=['erm']).sub(pc['erm'], axis=0).round(2))
ana = json.load(open(f'{F3}/class_analysis.json'))
for m, a in ana.items():
    print(f'\n{m}:')
    for tag in ['largest_gain', 'largest_drop']:
        e = a[tag]
        print(f'   {tag.replace("_", " "):13s}: {e["class"]:9s} {e["delta_pp"]:+.1f} pts  -> now confused with: '
              + ', '.join(f'{c["pred"]} ({c["share_%"]:.0f}%)' for c in e['confusions_after'][:2]))
print('\nSketch has only 80 house and 160 person images, so those two classes move in large jumps.')

In [ ]:
# D4. Research question 4: what did unlabelled Sketch data actually buy?
# Same MMD penalty, same kernels, same ERM baseline; the only difference is that Task 2's DAN saw
# unlabelled Sketch while Task 3's DAN-DG only aligned the three observed sources.
print(open(f'{F3}/task2_vs_task3_overall.json').read())
display(pd.read_csv(f'{F3}/task2_vs_task3_per_class.csv', index_col=0).round(1))
print('Limits on the causal claim: the two methods align different distributions (source-target vs')
print('source-source), over a different number of pairs, with different batch sizes per MMD estimate,')
print('and each is a single seed. Treat it as suggestive, not as a clean ablation of target access.')

In [ ]:
# D5. Figures.
from IPython.display import Image, display as show
CAPS = {'training_curves.png': 'Required: classification loss and the MMD penalty for the trained methods',
        'per_class_delta.png': 'Required: per-class Sketch accuracy change vs ERM'}
for p in sorted(glob.glob(f'{F3}/figures/*.png')):
    n = os.path.basename(p)
    print(f'\n--- {n} ---\n' + CAPS.get(n, 'Failure cases: correct under ERM, wrong after this method'))
    show(Image(p))

In [ ]:
# D6. Package everything (Tasks 1-3) for GitHub.
!rm -rf /tmp/gitcheck && git init -q /tmp/gitcheck
!git --git-dir=/tmp/gitcheck/.git --work-tree=. ls-files --others --exclude-standard > /tmp/commit_list.txt
print('files to commit:', len(open('/tmp/commit_list.txt').read().split()))
!du -ch $(cat /tmp/commit_list.txt) | tail -1
!rm -f "{DRIVE}/PA1_github.zip" && zip -q -@ "{DRIVE}/PA1_github.zip" < /tmp/commit_list.txt
print('Saved My Drive/PA1_github.zip')

## Troubleshooting
- **`Task 3 protocol differs from the ERM checkpoint`:** the check in A6 failed. Either revert `task2/configs/base.yaml` to the settings the Source-only checkpoint was trained with, or retrain Source-only under the current settings. Do not train Task 3 against a mismatched ERM.
- **A run is flagged COLLAPSED or unstable in B5:** send its `report()` output to Claude before evaluating.
- **`FileNotFoundError` on the ERM checkpoint:** Task 2's Source-only run must exist at `task2/results/runs/source_only/best.pt`.
- **Session died mid-run:** re-run Part A, then the same B cell. Finished runs are skipped.
- **SAM feels slow:** that is expected; it does two forward/backward passes per update.